# Chronos-Bolt Forecast - Sales Demand
Zero-shot forecasting using `amazon/chronos-bolt-small` on retail inventory dataset.

In [1]:
!pip install -q git+https://github.com/amazon-science/chronos-forecasting.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 133.4 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 14.1 MB/s eta 0:00:0000:01:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 15.5 MB/s eta 0:00:0000:01


In [2]:
import pandas as pd
import numpy as np
import torch
import time
import matplotlib.pyplot as plt
from chronos import ChronosBoltPipeline

import os

MODEL_ID  = "amazon/chronos-bolt-small"
TARGET    = "Units Sold"
HORIZONS  = [7, 14, 28]
TRAIN_END = '2023-06-30'
VAL_END   = '2023-10-31'
LAG       = 7

# Auto-detect environment
if os.path.exists('/kaggle/input'):
    import glob
    matches = glob.glob('/kaggle/input/**/sales_data.csv', recursive=True)
    DATA_PATH = matches[0] if matches else '/kaggle/input/sales_data.csv'
    RESULT_DIR = '/kaggle/working'
else:
    DATA_PATH  = '/Users/P837032/Daily/Model/dataset/sales_data.csv'
    RESULT_DIR = '/Users/P837032/Daily/Model/result'

print(f'DATA_PATH: {DATA_PATH}')

2026-03-30 12:51:54.890152: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774875115.150595      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774875115.234276      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774875115.947334      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774875115.947427      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774875115.947430      55 computation_placer.cc:177] computation placer alr

DATA_PATH: /kaggle/input/datasets/atomicd/retail-store-inventory-and-demand-forecasting/sales_data.csv


## 1. Load Data

In [3]:
df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
df["series_id"] = df["Store ID"] + "_" + df["Product ID"]
print(f"Shape: {df.shape}")
print(f"Date range: {df['Date'].min()} - {df['Date'].max()}")
print(f"Series count: {df['series_id'].nunique()}")
df.head()

Shape: (76000, 17)
Date range: 2022-01-01 00:00:00 - 2024-01-30 00:00:00
Series count: 100


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,Weather Condition,Promotion,Competitor Pricing,Seasonality,Epidemic,Demand,series_id
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,Snowy,0,85.73,Winter,0,115,S001_P0001
1,2022-01-01,S001,P0002,Clothing,North,117,117,249,80.16,15,Snowy,1,92.02,Winter,0,229,S001_P0002
2,2022-01-01,S001,P0003,Clothing,North,247,114,612,62.94,10,Snowy,1,60.08,Winter,0,157,S001_P0003
3,2022-01-01,S001,P0004,Electronics,North,139,45,102,87.63,10,Snowy,0,85.19,Winter,0,52,S001_P0004
4,2022-01-01,S001,P0005,Groceries,North,152,65,271,54.41,0,Snowy,0,51.63,Winter,0,59,S001_P0005


## 2. Metrics

In [4]:
def mase(y, p, train):
    lag = min(LAG, len(train) - 1)
    denom = np.mean(np.abs(train[lag:] - train[:-lag]))
    return np.mean(np.abs(y - p)) / denom if denom > 0 else np.nan
def smape(y, p):
    return np.mean(2 * np.abs(y - p) / (np.abs(y) + np.abs(p) + 1e-8)) * 100

def rmse(y, p):
    return float(np.sqrt(np.mean((y - p) ** 2)))

def rmsle(y, p):
    fc_c = np.clip(p, 0, None)
    ac_c = np.clip(y, 0, None)
    return float(np.sqrt(np.mean((np.log1p(fc_c) - np.log1p(ac_c)) ** 2)))


## 3. Load Model

In [5]:
pipeline = ChronosBoltPipeline.from_pretrained(MODEL_ID, device_map="cpu", torch_dtype=torch.float32)
print('Model loaded.')

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/191M [00:00<?, ?B/s]

Model loaded.


## 4. Rolling Eval per Store × Product × Horizon

In [6]:
df['series_id'] = df['Store ID'] + '_' + df['Product ID']
series_ids = sorted(df['series_id'].unique())
os.makedirs(RESULT_DIR, exist_ok=True)

results = []
details = []

for h in HORIZONS:
    print(f'\n=== Horizon = {h} ===')
    scores = {'mase': [], 'smape': [], 'rmse': [], 'rmsle': []}

    # Build contexts từ VAL_END trở về trước (dùng làm train context)
    contexts, actuals_list, trains_list = [], [], []
    valid_ids = []
    eval_start = pd.Timestamp(VAL_END) + pd.Timedelta(days=1)

    for sid in series_ids:
        sdf = df[df['series_id'] == sid].sort_values('Date').set_index('Date')
        ts_train = sdf[:VAL_END][TARGET].values.astype(np.float32)
        ts_test  = sdf[eval_start:][TARGET].values.astype(np.float32)
        if len(ts_test) < h or len(ts_train) < 2:
            continue
        contexts.append(torch.tensor(ts_train))
        actuals_list.append(ts_test[:h])
        trains_list.append(ts_train)
        valid_ids.append(sid)

    start = time.time()
    forecasts = pipeline.predict(contexts, prediction_length=h)
    preds = forecasts.median(dim=1).values.numpy()
    print(f'  Inference: {time.time()-start:.1f}s for {len(valid_ids)} series')

    for i, sid in enumerate(valid_ids):
        store, product = sid.split('_', 1)
        y, p, tr = actuals_list[i], preds[i], trains_list[i]
        r = {'mase': mase(y, p, tr), 'smape': smape(y, p), 'rmse': rmse(y, p), 'rmsle': rmsle(y, p)}
        for k in scores: scores[k].append(r[k])
        details.append({'model': 'Chronos-Bolt-Small', 'store': store, 'product': product, 'horizon': h,
                        'mase': round(float(r['mase']), 4), 'smape': round(float(r['smape']), 4),
                        'rmse': round(float(r['rmse']), 4), 'rmsle': round(float(r['rmsle']), 4)})
        print(f"  {store} | {product} | MASE={r['mase']:.4f} sMAPE={r['smape']:.2f}%")

    row = {
        'model': 'Chronos-Bolt-Small', 'dataset': 'retail_inventory_daily',
        'target': TARGET, 'horizon': h,
        'mean_mase':    round(float(np.nanmean(scores['mase'])),    4),
        'median_mase':  round(float(np.nanmedian(scores['mase'])),  4),
        'mean_smape':   round(float(np.nanmean(scores['smape'])),   4),
        'median_smape': round(float(np.nanmedian(scores['smape'])), 4),
        'mean_rmse':    round(float(np.nanmean(scores['rmse'])),    4),
        'median_rmse':  round(float(np.nanmedian(scores['rmse'])),  4),
        'mean_rmsle':   round(float(np.nanmean(scores['rmsle'])),   4),
        'median_rmsle': round(float(np.nanmedian(scores['rmsle'])), 4),
    }
    results.append(row)
    print(f"  H={h} | MASE={row['mean_mase']:.4f} sMAPE={row['mean_smape']:.2f}% RMSE={row['mean_rmse']:.2f}")

summary = pd.DataFrame(results)
summary


=== Horizon = 7 ===
  Inference: 2.1s for 100 series
  S001 | P0001 | MASE=0.7427 sMAPE=28.01%
  S001 | P0002 | MASE=0.6356 sMAPE=32.91%
  S001 | P0003 | MASE=0.3473 sMAPE=18.68%
  S001 | P0004 | MASE=0.4962 sMAPE=24.60%
  S001 | P0005 | MASE=0.4724 sMAPE=23.33%
  S001 | P0006 | MASE=0.5213 sMAPE=46.55%
  S001 | P0007 | MASE=0.9702 sMAPE=43.44%
  S001 | P0008 | MASE=0.5729 sMAPE=26.94%
  S001 | P0009 | MASE=1.0340 sMAPE=45.23%
  S001 | P0010 | MASE=0.6000 sMAPE=24.72%
  S001 | P0011 | MASE=0.4325 sMAPE=27.69%
  S001 | P0012 | MASE=0.9630 sMAPE=39.83%
  S001 | P0013 | MASE=0.8372 sMAPE=36.63%
  S001 | P0014 | MASE=0.6240 sMAPE=27.72%
  S001 | P0015 | MASE=0.8344 sMAPE=44.95%
  S001 | P0016 | MASE=0.3822 sMAPE=19.29%
  S001 | P0017 | MASE=0.4404 sMAPE=25.63%
  S001 | P0018 | MASE=0.8433 sMAPE=41.70%
  S001 | P0019 | MASE=0.6650 sMAPE=28.66%
  S001 | P0020 | MASE=0.3916 sMAPE=22.72%
  S002 | P0001 | MASE=0.8899 sMAPE=34.51%
  S002 | P0002 | MASE=0.7968 sMAPE=42.83%
  S002 | P0003 | MASE=

,model,dataset,target,horizon,mean_mase,median_mase,mean_smape,median_smape,mean_rmse,median_rmse,mean_rmsle,median_rmsle
0,Chronos-Bolt-Small,retail_inventory_daily,Units Sold,7,0.7144,0.6449,34.3359,32.7843,36.9540,33.7793,0.4563,0.4038
1,Chronos-Bolt-Small,retail_inventory_daily,Units Sold,14,0.7292,0.6884,35.0910,33.1940,38.2524,35.8689,0.4930,0.4159
2,Chronos-Bolt-Small,retail_inventory_daily,Units Sold,28,0.7427,0.7149,35.9839,35.5498,39.5369,37.6004,0.5182,0.4544


## 5. Save & Compare

In [7]:
out_path = f'{RESULT_DIR}/chronos_bolt_daily_summary.csv'
summary.to_csv(out_path, index=False)
pd.DataFrame(details).to_csv(f'{RESULT_DIR}/chronos_bolt_daily_details.csv', index=False)
print(f'Saved to {out_path}')

# So sánh với baseline nếu có
baseline_path = f'{RESULT_DIR}/daily_baseline_summary.csv'
if os.path.exists(baseline_path):
    baseline = pd.read_csv(baseline_path)
    comparison = pd.concat([baseline, summary], ignore_index=True)
    display(comparison[['model','horizon','mean_mase','median_mase']])
else:
    display(summary[['model','horizon','mean_mase','median_mase']])

Saved to /kaggle/working/chronos_bolt_daily_summary.csv


,model,horizon,mean_mase,median_mase
0,Chronos-Bolt-Small,7,0.7144,0.6449
1,Chronos-Bolt-Small,14,0.7292,0.6884
2,Chronos-Bolt-Small,28,0.7427,0.7149
